# Cuaderno de arranque: el almacén de datos (capa oro)

Este cuaderno no hace ningún análisis. Sirve para empezar: conecta con el almacén, enseña qué hay dentro y cómo se cruzan las tablas, y deja preparados los datos de partida de cada análisis del Bloque 2 (RFM, previsión, reglas de asociación y embudo).

**Antes de empezar**

- Abre el cuaderno desde la carpeta que contiene la carpeta `oro`.
- Hace falta DuckDB: `pip install duckdb` (una sola vez).
- No regeneres la carpeta `oro` por tu cuenta: los códigos de las personas seudonimizadas dependen de una clave que guarda Rubén. Si hace falta regenerarla, pídeselo.

**Tres cosas que conviene tener presentes**

1. **Importes en soles y sin IGV.** Cada importe está también en su moneda original (`*_doc`); las columnas `*_pen` ya están convertidas.
2. **Las notas de crédito restan.** En `fact_ventas` tienen importe negativo, así que una suma da directamente las ventas netas.
3. **Las personas están seudonimizadas.** Los clientes con DNI aparecen como `PN-…` y el personal como `VEN-01`, `USR-01`. CLIENTES VARIOS (`00000000`) es el mostrador sin identificar: no es un cliente.

## 1. Conexión

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

ORO = Path('oro')
assert (ORO / 'almacen.duckdb').exists(), 'No encuentro oro/almacen.duckdb: abre el cuaderno desde la carpeta que contiene la carpeta oro'

# Solo lectura: desde el cuaderno no se puede modificar el almacén por error
con = duckdb.connect(str(ORO / 'almacen.duckdb'), read_only=True)


def q(sql):
    """Ejecuta una consulta SQL sobre el almacén y devuelve un DataFrame de pandas."""
    return con.sql(sql).df()

## 2. Qué hay dentro

Las tablas `fact_…` son los **hechos**: lo que ha pasado (cada línea vendida, cada proforma, cada cobro), con lo que se mide (importes, cantidades, días). Las tablas `dim_…` son las **dimensiones**: el quién, qué y cuándo con el que se filtra y agrupa (cliente, producto, fecha, vendedor...). Las vistas `v_…` son los indicadores del catálogo ya calculados.

In [ ]:
q("""
    SELECT table_name AS nombre, 'tabla' AS tipo, estimated_size AS filas FROM duckdb_tables()
    UNION ALL
    SELECT view_name, 'vista', NULL FROM duckdb_views() WHERE NOT internal
    ORDER BY tipo, nombre
""")

El significado de cada columna está en el diccionario de datos. Por ejemplo, las de `fact_ventas`:

In [ ]:
diccionario = pd.read_csv(ORO / 'diccionario_datos.csv')
pd.set_option('display.max_colwidth', 120)
diccionario.query("TABLA == 'fact_ventas'")[['COLUMNA', 'DESCRIPCION']]

## 3. Cómo se cruzan hechos y dimensiones

Los hechos guardan claves (`…_key`) que apuntan a una fila de cada dimensión. Para obtener las ventas por mes se une `fact_ventas` con `dim_fecha` por la fecha de emisión y se suma:

In [ ]:
ventas_mes = q("""
    SELECT f.anio_mes, ROUND(SUM(v.importe_neto_pen), 2) AS ventas_netas_pen
    FROM fact_ventas v
    JOIN dim_fecha f ON f.fecha_key = v.fecha_emision_key
    GROUP BY f.anio_mes
    ORDER BY f.anio_mes
""")
ventas_mes.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(pd.to_datetime(ventas_mes.anio_mes), ventas_mes.ventas_netas_pen / 1000, color='#2a78d6', linewidth=2)
ax.set_title('Ventas netas por mes: crecen y marcan picos en diciembre', loc='left', fontsize=12)
ax.set_ylabel('miles de S/ (sin IGV)')
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))
ax.set_ylim(bottom=0)
ax.grid(axis='y', color='#e5e5e5', linewidth=0.8)
ax.spines[['top', 'right']].set_visible(False)
plt.show()

## 4. Lo mismo, con una vista

Las vistas ya hacen esos cruces. `v_ventas_mensuales` da las ventas, el margen, el ticket medio, los clientes y las notas de crédito de cada mes, calculados como dice el catálogo de indicadores. Conviene partir de ellas: así todos calculamos igual.

In [ ]:
vm = q('SELECT * FROM v_ventas_mensuales')
assert (vm.ventas_netas_pen.values == ventas_mes.ventas_netas_pen.values).all()  # coincide con el cálculo a mano
vm.tail()

In [ ]:
# El catálogo de indicadores viaja dentro del almacén
q('SELECT codigo, indicador, fuente, valor_referencia FROM catalogo_indicadores')

## 5. Datos de partida para el Bloque 2

Cada celda deja preparada una tabla para un análisis. No hacen el análisis: solo ahorran el trabajo de sacar los datos.

### 5.1 Segmentación de clientes (RFM)

Una fila por cliente: días desde su última compra (recencia), número de compras (frecuencia) e importe neto (valor monetario), con sus atributos actuales. La fecha de referencia es la última venta registrada.

In [ ]:
rfm = q("""
    WITH ventas AS (
        SELECT c.cliente_id, CAST(f.fecha AS DATE) AS fecha, v.serie || '-' || v.numero AS documento,
               v.importe_neto_pen, td.signo
        FROM fact_ventas v
        JOIN dim_cliente c         ON c.cliente_key = v.cliente_key
        JOIN dim_fecha f           ON f.fecha_key = v.fecha_emision_key
        JOIN dim_tipo_documento td ON td.tipo_documento_key = v.tipo_documento_key
        WHERE c.cliente_id <> '00000000'
    )
    SELECT v.cliente_id,
           DATE_DIFF('day', MAX(v.fecha) FILTER (WHERE v.signo = 1), (SELECT MAX(fecha) FROM ventas)) AS recencia_dias,
           COUNT(DISTINCT v.documento) FILTER (WHERE v.signo = 1)                                 AS frecuencia,
           ROUND(SUM(v.importe_neto_pen), 2)                                                      AS monetario_pen,
           ANY_VALUE(c.tipo_cliente) AS tipo_cliente, ANY_VALUE(c.sector) AS sector,
           ANY_VALUE(c.departamento) AS departamento, ANY_VALUE(c.condicion_credito) AS condicion_credito
    FROM ventas v
    JOIN dim_cliente c ON c.cliente_id = v.cliente_id AND c.es_actual
    GROUP BY v.cliente_id
""")
print(len(rfm), 'clientes')
rfm.describe().round(1)

### 5.2 Previsión de ventas

La serie mensual de ventas netas, con los últimos seis meses reservados para validar el modelo (la Tabla 5 pide medir el error sobre un periodo reservado). Con 32 meses hay menos de tres ciclos anuales: el periodo reservado no debería ser más largo.

In [ ]:
serie = vm.set_index(pd.PeriodIndex(vm.anio_mes, freq='M')).ventas_netas_pen
entrenamiento, prueba = serie[:-6], serie[-6:]
print(f'Entrenamiento: {entrenamiento.index[0]} a {entrenamiento.index[-1]} ({len(entrenamiento)} meses)')
print(f'Prueba:        {prueba.index[0]} a {prueba.index[-1]} ({len(prueba)} meses)')

### 5.3 Reglas de asociación

Una fila por factura o boleta y una columna por familia (o por producto), con `True` si el documento la incluye. Es el formato que espera `mlxtend` (`apriori`, `association_rules`).

In [ ]:
lineas = q("""
    SELECT v.serie || '-' || v.numero AS documento, p.familia, p.codigo_producto
    FROM fact_ventas v
    JOIN dim_producto p        ON p.producto_key = v.producto_key
    JOIN dim_tipo_documento td ON td.tipo_documento_key = v.tipo_documento_key
    WHERE td.signo = 1
""")
cestas_familia = pd.crosstab(lineas.documento, lineas.familia).gt(0)
cestas_producto = pd.crosstab(lineas.documento, lineas.codigo_producto).gt(0)
print(cestas_familia.shape, 'documentos x familias |', cestas_producto.shape, 'documentos x productos')
cestas_familia.mean().sort_values(ascending=False).head(8).round(3)  # parte de documentos que incluye cada familia

### 5.4 Embudo comercial

Una fila por proforma, con su vendedor, el tipo de cliente, el estado y si acabó en venta. Es la base para estudiar por qué se pierden proformas.

In [ ]:
embudo = q("""
    SELECT e.num_proforma, CAST(f.fecha AS DATE) AS fecha_emision, vd.vendedor_codigo,
           c.tipo_cliente, c.sector, ep.estado, e.convertida, e.dias_hasta_venta,
           e.importe_ofertado_pen, e.importe_facturado_pen
    FROM fact_embudo_proforma e
    JOIN dim_fecha f            ON f.fecha_key = e.fecha_emision_key
    JOIN dim_vendedor vd        ON vd.vendedor_key = e.vendedor_key
    JOIN dim_cliente c          ON c.cliente_key = e.cliente_key
    JOIN dim_estado_proforma ep ON ep.estado_proforma_key = e.estado_proforma_key
""")
resueltas = embudo[embudo.estado != 'ABIERTA']
resueltas.groupby('vendedor_codigo').convertida.mean().mul(100).round(1).rename('conversion_%')

## Siguientes pasos

- La definición de cada indicador está en el documento *Catálogo de indicadores*; el porqué del modelo, en *Capa oro: el modelo en estrella*.
- Si un análisis necesita un dato que no está en el almacén, mejor pedir que se añada en `oro.py` que calcularlo aparte: así lo tiene también el cuadro de mando.
- No guardes resultados dentro de la carpeta `oro`: se regenera entera cada vez que se ejecuta `oro.py`.